# RedKite Aerospace - Hand-Signal Gesture Classification for Autonomous Medical Delivery

## 1. Background

RedKite Aerospace is pursuing a contract with a regional emergency-services provider to deliver
time-critical medical supplies (e.g. blood products) via drone to sites that are difficult for
ground vehicles to reach quickly. The client's Request for Proposal (RFP) mandates that, on
arrival at a delivery site, the aircraft must be able to interpret marshalling-style hand
signals from the person receiving the package — with no radio link and no companion app. The
receiver may direct the drone left or right, ask it to hold, signal that the landing zone is
clear or unsafe, or wave the aircraft off entirely.

RedKite's airframe already satisfies every hardware requirement in the RFP. This gesture-reading
capability is the one missing piece, and a live demonstration is scheduled in three weeks. If the
aircraft can correctly read a client staff member's gestures on the day, the contract is
winnable; if not, the opportunity is lost.

## 2. Task

An onboard segmenter (provided, out of scope for this notebook) observes the receiver and splits
the video stream into discrete gesture **instances**. Each instance is delivered to the
classifier as exactly **5 frames**.

**Goal:** given one pre-segmented instance (5 frames), classify it into one of 13 gesture
classes:

`AllClear`, `HaveCommand`, `Hover`, `Land`, `LandingDirection`, `MoveAhead`, `MoveDownward`,
`MoveToLeft`, `MoveToRight`, `MoveUpward`, `NotClear`, `SlowDown`, `WaveOff`

This notebook covers **classification only** — no person detection, no video segmentation, and
no live-stream handling. One instance in, one class out.

## 3. Constraints and Operating Context

- **Timeline:** three weeks to a live demo — favours pragmatic, well-justified choices over
  exhaustive experimentation.
- **Data:** a single afternoon of colleagues (non-professional signallers) performing the
  gestures on camera. Small sample size, limited signaller diversity, and likely class imbalance
  are expected and will shape both modelling choices and how much the results can be trusted.
- **Deliverable expectation:** the executives are not just looking for a working model — they
  want sound engineering judgement and an honest account of how far the result generalises
  beyond this dataset (e.g. to a different signaller, lighting, or background on demo day).

## 4. Notebook Structure

1. Data loading and exploration
2. Preprocessing and augmentation strategy
3. Model design and training
4. Evaluation (including class-level performance and error analysis)
5. Discussion of limitations and confidence in real-world (demo-day) performance

In [1]:
# Loading necessary libraries
from pathlib import Path
import pandas as pd
import re

In [2]:
# Dataset Loading
DATASET_ROOT = Path("data_resized")  # <-- adjust to wherever the dataset lives in the notebook space

ACTIONS = [
    "AllClear", "HaveCommand", "Hover", "Land", "LandingDirection",
    "MoveAhead", "MoveDownward", "MoveToLeft", "MoveToRight",
    "MoveUpward", "NotClear", "SlowDown", "WaveOff"
]

records = []

for action in ACTIONS:
    action_dir = DATASET_ROOT / action
    if not action_dir.exists():
        print(f"Missing folder for action: {action}")
        continue

    for subject_dir in sorted(action_dir.iterdir()):
        if not subject_dir.is_dir():
            continue
        subject_id = subject_dir.name  # e.g. "S1"

        for instance_dir in sorted(subject_dir.iterdir()):
            if not instance_dir.is_dir():
                continue
            instance_id = instance_dir.name  # e.g. "instance1"

            frame_paths = sorted(instance_dir.glob("*.png"))

            records.append({
                "action": action,
                "subject": subject_id,
                "instance": instance_id,
                "n_frames": len(frame_paths),
                "frame_paths": [str(p) for p in frame_paths],
            })

df = pd.DataFrame(records)
print(f"Total instances found: {len(df)}")
df.head()

Total instances found: 1059


,action,subject,instance,n_frames,frame_paths
0,AllClear,S1,Instance1,5,[data_resized/AllClear/S1/Instance1/S1_allClea...
1,AllClear,S1,Instance2,5,[data_resized/AllClear/S1/Instance2/S1_allClea...
2,AllClear,S1,Instance3,5,[data_resized/AllClear/S1/Instance3/S1_allClea...
3,AllClear,S1,Instance4,5,[data_resized/AllClear/S1/Instance4/S1_allClea...
4,AllClear,S1,Instance5,5,[data_resized/AllClear/S1/Instance5/S1_allClea...


In [3]:
# Every instance should have exactly 5 frames
bad = df[df["n_frames"] != 5]
if len(bad) > 0:
    print(f"{len(bad)} instances do NOT have 5 frames:")
    display(bad[["action", "subject", "instance", "n_frames"]])
else:
    print("All instances have exactly 5 frames.")

# Class balance
print("\nInstances per action:")
print(df["action"].value_counts())

# Subjects available
print("\nSubjects present:")
print(sorted(df["subject"].unique()))

# Instances per subject per action
pd.crosstab(df["subject"], df["action"])

All instances have exactly 5 frames.

Instances per action:
action
Land                122
MoveAhead            99
MoveToLeft           95
MoveToRight          95
SlowDown             93
AllClear             87
NotClear             86
HaveCommand          85
Hover                82
MoveUpward           62
WaveOff              59
LandingDirection     48
MoveDownward         46
Name: count, dtype: int64

Subjects present:
['S1', 'S11', 'S12', 'S13', 'S14', 'S15', 'S3', 'S4', 'S5', 'S6', 'S9']


action,AllClear,HaveCommand,Hover,Land,LandingDirection,MoveAhead,MoveDownward,MoveToLeft,MoveToRight,MoveUpward,NotClear,SlowDown,WaveOff
subject,,,,,,,,,,,,,
S1,8,9,9,17,7,10,8,9,9,10,5,9,9
S11,9,7,9,19,9,8,5,9,8,9,9,9,9
S12,8,7,11,18,8,10,8,8,9,9,7,10,12
S13,4,9,11,18,8,9,5,9,8,10,9,7,8
S14,10,8,0,0,0,8,0,9,8,0,10,11,0
S15,7,9,0,0,0,10,0,9,9,0,9,10,0
S3,9,10,0,0,0,10,0,10,9,0,10,9,0
S4,7,7,13,17,7,6,7,7,10,8,6,6,7
S5,6,5,16,17,4,8,6,7,6,7,6,5,6
